# Data Synthesizer: 3-Phase Motor Signals
### **Objective:** Generate a 10kHz synthetic dataset matching the CARDD-Tech lab's expected CSV format (`Va, Vb, Vc, Ia, Ib, Ic, label`) and save it securely to Google Drive.
***

# Cell 1: Environment Setup & Cloud Mounting

In [1]:
import os
import numpy as np
import pandas as pd

# Mount Google Drive to save the dataset persistently
try:
    from google.colab import drive
    print("[INFO] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("❌ Error: Not running in a Colab environment! This script requires Colab.")

# Define your exact Google Drive path
project_root = '/content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline'
data_folder = f'{project_root}/data'
os.makedirs(data_folder, exist_ok=True)
print(f"✅ Data routing established at: {data_folder}")

[INFO] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Data routing established at: /content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline/data


***
# Cell 2: Generate 10kHz Synthetic Motor Data

In [2]:
print("[INFO] Generating synthetic 3-phase AC signals...")

# Parameters
num_samples = 10000
time = np.linspace(0, 1, num_samples) # 1 second of data at 10kHz

# Generate clean 3-phase AC signals (120 degrees apart)
voltage_u = 120 * np.sin(2 * np.pi * 60 * time)
voltage_v = 120 * np.sin(2 * np.pi * 60 * time - (2 * np.pi / 3))
voltage_w = 120 * np.sin(2 * np.pi * 60 * time + (2 * np.pi / 3))

current_u = 15 * np.sin(2 * np.pi * 60 * time)
current_v = 15 * np.sin(2 * np.pi * 60 * time - (2 * np.pi / 3))
current_w = 15 * np.sin(2 * np.pi * 60 * time + (2 * np.pi / 3))

# Add realistic sensor noise
noise_level = 0.5
df = pd.DataFrame({
    'Va': voltage_u + np.random.normal(0, noise_level, num_samples),
    'Vb': voltage_v + np.random.normal(0, noise_level, num_samples),
    'Vc': voltage_w + np.random.normal(0, noise_level, num_samples),
    'Ia': current_u + np.random.normal(0, noise_level, num_samples),
    'Ib': current_v + np.random.normal(0, noise_level, num_samples),
    'Ic': current_w + np.random.normal(0, noise_level, num_samples),
    'label': 0 # 0 = Normal operating condition
})

[INFO] Generating synthetic 3-phase AC signals...


***
# Cell 3: Inject Anomalies & Save

In [3]:
print("[INFO] Injecting synthetic stator short circuit anomaly...")

# Inject a synthetic anomaly (stator short circuit spike at sample 8000)
df.loc[8000:8050, 'Ia'] += 25.0
df.loc[8000:8050, 'label'] = 1 # 1 = Fault detected

# Save directly to the cloud
file_path = f'{data_folder}/raw_motor_signals.csv'
df.to_csv(file_path, index=False)

print(f"✅ Successfully created and saved {num_samples} rows to Google Drive at: {file_path}")

# Display the first few rows to verify the headers
df.head()

[INFO] Injecting synthetic stator short circuit anomaly...
✅ Successfully created and saved 10000 rows to Google Drive at: /content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline/data/raw_motor_signals.csv


,Va,Vb,Vc,Ia,Ib,Ic,label
0,0.604841,-103.725003,104.294220,0.106806,-12.759244,13.094915,0
1,4.242903,-106.680701,101.624604,1.882376,-13.006116,12.598877,0
2,8.563598,-108.324336,99.701838,1.121591,-12.457059,11.137581,0
3,13.571636,-109.709957,97.153791,1.926710,-13.548250,11.217281,0
4,18.214445,-111.205494,93.681807,2.873038,-14.228192,11.485647,0
